In [ ]:
# ==========================================
# Built-in Python Modules
# ==========================================

import sys   # Access Python interpreter information (version, path, arguments, etc.)
import os    # Interact with the operating system (files, folders, environment variables)


# ==========================================
# LLM (Ollama) Integration
# ==========================================

from langchain_ollama import ChatOllama
# Connect and chat with local Ollama models (Llama, Mistral, Gemma, etc.)


# ==========================================
# External Tools
# ==========================================

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
# Search and retrieve information from Wikipedia

from langchain_community.tools import DuckDuckGoSearchRun
# Search the web using DuckDuckGo


# ==========================================
# Custom Tool Creation
# ==========================================

from langchain_core.tools import tool
# Convert Python functions into LangChain tools using @tool


# ==========================================
# Prompt Templates
# ==========================================

from langchain_core.prompts import ChatPromptTemplate
# Create reusable prompts with dynamic variables


# ==========================================
# ReAct Agent ## use lang-graph
# ==========================================

from langgraph.prebuilt import create_react_agent
# Create a ReAct (Reason + Act) agent that can use tools


# ==========================================
# Conversation Memory
# ==========================================

from langgraph.checkpoint.memory import InMemorySaver
# Store conversation history in memory for continuous chatting




In [ ]:
# ChatOllama      ← LangChain
# Wikipedia Tool  ← LangChain
# DuckDuckGo Tool ← LangChain
# Prompt          ← LangChain

# create_react_agent ← LangGraph
# InMemorySaver      ← LangGraph
# thread_id          ← LangGraph

In [4]:
print(sys.version)
print(os.getcwd())

3.14.5 (tags/v3.14.5:5607950, May 10 2026, 10:43:50) [MSC v.1944 64 bit (AMD64)]
d:\github\Learning\RAG\notebook


In [ ]:
print("🔄 Initializing Ollama (Llama 3.1)...")
llm = ChatOllama(
    model="llama3.1:8b",
    base_url="http://localhost:11434",
    temperature=0.3,
    timeout=60  
)
# ==========================================
# 2. Tools Configuration
# ==========================================
print("📦 Loading search and math tools...")
wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
search_tool = DuckDuckGoSearchRun()

@tool
def calculator(expression: str) -> str:
    """Use this tool for math calculations like 2+2, 10*5, etc."""
    try:
        allowed_chars = "0123456789+-*/(). "
        if not all(char in allowed_chars for char in expression):
            return "Error: Invalid characters in math expression."
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

tools = [wiki, search_tool, calculator]
# ==========================================
# 3. Create Modern ReAct Agent with Checkpointer
# ==========================================
print("🤖 Compiling ReAct Agent Graph...")

# Using InMemorySaver handles conversation thread states automatically
memory = InMemorySaver()

# Create a proper prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "You are a helpful assistant. Think step-by-step. "
     "Use your search, wikipedia, and calculator tools whenever you need external or precise information."),
    ("placeholder", "{messages}")
])

# Use create_react_agent with the prompt parameter instead of system_prompt
agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=prompt,  # ✅ Use 'prompt' instead of 'system_prompt'
    checkpointer=memory,
    debug=False  # Toggle to True if you want to inspect low-level LangGraph execution traces
)

In [ ]:
UserID = "user1"

config = {
    "configurable": {
        "thread_id": UserID
    }
}

# ==========================================
# 4. Interactive Jupyter Function
# ==========================================
def ask_agent(user_input: str):
    """Call this function inside any Jupyter cell to talk to the agent."""
    if not user_input.strip():
        return
        
    print(f"You: {user_input}")
    print("Thinking...")
    
    try:
        # Pass the message payload structure expected by create_react_agent
        result = agent.invoke(
            {"messages": [{"role": "user", "content": user_input}]}, 
            config=config
        )
        
        # Pull the absolute last response generated in the message graph state
        ai_response = result["messages"][-1].content
        #ai_response = result["messages"]
        print(f"\nAI: {ai_response}\n")
        #print(f"\nMemory: {memory}\n")
    except Exception as e:
        print(f"\nExecution Error: {e}\n")

print("\n🚀 Setup Complete! You can now converse with the agent using the ask_agent() function.")